In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-14: Can I trust the comparison? Checks

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-13's map and rule say what to try. They cannot say whether a comparison, a teacher or a reward can be trusted. This session: four checks, and the close of Part 2.**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples, 2 who supplies the weights, 3 how fine. No new method: we check the ones placed
- **Position**: session 14 of 14, after RLLM-13 (The map and the decision tool)
- **Goal of Part 2**: a **map**, and the checks that say when to trust a comparison

<img src="images/series_map_14.png" width=55%>


## This session: RLLM-14

- **Question**: how do you **check** a comparison, a teacher and a **reward** before you trust them?
- **By the end you can**:
  - say why **one run** is not a comparison, and compare a gap with the **run-to-run noise** (seed-to-seed or re-evaluation)
  - check for **collapse**: diversity, length, format, and an **input-blind baseline**
  - test a **teacher** with a **struggle set** and a **planted-error probe**
  - probe a **reward** with a wrong-but-passing list and a right-but-failing list


## Map: four doubts, four checks

| Before you rely on... | Try | Section |
|---|---|---|
| a **gap** between two runs | several seeds each; gap against spread | Noise |
| an **average** that rose | diversity, length, format, per-case view; an input-blind baseline (a policy that ignores the prompt) | Collapse |
| a **teacher** | a struggle-set comparison; a planted-error probe, with and without context | Teacher |
| a **reward** (a function, a verifier, a reward model) | wrong-but-passing and right-but-failing lists | Reward |

- Each check is a **suggestion**: weigh its cost against a training run
- They say whether a comparison **can** be trusted. We rank nothing
- *Next: the noise a gap must beat*


## Noise: three kinds, and which comparison each fits

**Why this section?** A gap is judged against noise. Which noise depends on what you compare.

| Kind of noise | How to see it | Applies to |
|---|---|---|
| **seed-to-seed**: retrain with another seed | several seeds each | comparing **methods** |
| **re-evaluation**: score the **same** trained model again | repeat the scoring | whether one model's score is stable (greedy decoding: zero) |
| **prompt-set**: the prompt set is finite | a paired bootstrap on the **same** prompts (RLLM-12) | comparing two models on one prompt set |

- **Run-to-run noise** (RLLM-12's name) is the first two rows
- Prompt-set standard error (RLLM-12), $n=200$ prompts, 60%:
  - one accuracy: about **3.5 points**; a gap between two accuracies: about **5**
- Always say **how many** seeds
- *Next: the rule a gap must pass*

## Noise: a first check, the ranges

- **A first check, not a test**: do the two methods' **ranges overlap**?
  - **no overlap**: look closer; with few seeds it is **not proof**
  - **overlap**: you **cannot tell yet**; it does **not** mean equal
- A **seed** changes the sampling, the initialization and the data order; the evaluation set stays
- One run per method **confounds** method and seed
- *Next: the rule that a gap must exceed the noise*


## Noise: a gap counts only if it exceeds the noise

> **From RLLM-12: a teacher is unlikely to help unless it beats the student by more than the noise (our inference). Prompt-set noise (a gap: about 5 points, RLLM-12) and run-to-run noise both count. The same test fits any two methods.**
<!-- seed-pay: quality-floor -->

<img src="images/seed_spread_toy.png" alt="two methods with the same mean gap; with a small run-to-run spread the gap stands out, with a large spread it is buried" width=70%>

- Both panels have the **same mean gap** (invented data)
  - **left**: the runs barely move, so the gap **stands out**
  - **right**: the runs move more than the gap, so it is **buried**
- *Next: a worked example with three seeds each*


## Noise: a worked example, three seeds each (invented)

- Method A beats B by a mean gap of **3** points
- Each method has **3** seeds and a spread of **s = 2**
- The standard error of the difference of two means: $\sqrt{2s^2/n}=\sqrt{2\cdot 4/3}\approx 1.6$
- The gap is about **1.8** standard errors: **not yet distinguishable**; we would want several
  - **rule of thumb** (our reading): a gap under about 2 standard errors is not yet distinguishable
  - the aside after the closing gives the formula
- This toy has seed noise only; with a fixed prompt set, prompt-set noise also counts
- *Next: what an average can hide*


## Collapse: what the average hides, collapse checks

**Why this section?** An average reward can rise while the model narrows. Two earlier questions need a measurement.

<img src="images/collapse_checks.png" alt="average reward rises over training steps while distinct completions per prompt fall" width=70%>

- Average reward rises while the model **narrows** (invented numbers)
- Check **at several points** in training:
  - **diversity**: distinct completions per prompt, by **exact match** of the text, or the **normalized final answer**
    - RLLM-9 asked what repeated filtering does to variety, RLLM-11 what each KL order does
  - the share of prompts whose completions **all earn the same reward**: they teach nothing to a group-relative method (RLLM-4)
- *Next: more checks that look past the average*

## Collapse: more checks that look past the average

- **length**, **format rate**, accuracy **per prompt type**
- a **per-case view**: which prompts got better and which got worse, not only the mean
- for DPO (RLLM-10):
  - how **stale** the samples are (from an older model, RLLM-7)
  - whether the winner's own log-probability **falls** during training
- *Next: is the gain about the input?*


Back to RLLM-9's all-equal groups (all-0 and all-1): which filter keeps what? Check your own answer to that Think-about.

## Collapse: is the gain about the input? An input-blind baseline

- **Reason**: an average reward can rise because the model learned what the **reward** likes (a fixed answer, a format), not the task
- **Input-blind baseline**: a policy that **ignores the input**
  - one fixed answer for every prompt, or the answers **shuffled** (each prompt gets another's)
- **Reading**: the part of the trained model's reward that the **input** explains is its reward minus this baseline's
  - if the baseline scores close to the trained model, the gain is **not** about the input
- Cheap: no training, one reward call per prompt
- *Next: checking a teacher*


## Teacher: is it informative where the student struggles?

**Why this section?** RLLM-12 named the struggle set; now we test it.

**Regression to the mean**: students picked for a low score usually score higher when re-tested. So the student's first score there is too low.

- So re-evaluate the student on fresh attempts, and compare the teacher with that. A toy (invented), 100 prompts:

| Prompts | Student | Teacher |
|---|---|---|
| half A: the student's struggle set | 20% re-evaluated (first score 0%, by construction) | 20% |
| half B | 90% | 100% |
| all 100 | 55% | 60% |

- The average says the teacher is better. On the struggle set it adds **nothing** (our reading)
- Read that gap against that set's own **prompt-set** noise (a smaller set, so a larger noise), as in RLLM-12's Test 2
- *Next: what the teacher does at the student's mistakes*


## Teacher: what does it do at the student's mistakes?

**Why this probe?** RLLM-12's premise (a teacher repairs a mistake) is untested. This probe tests it, per teacher, task and context.

<img src="images/teacher_at_mistake.png" alt="the teacher reads a student prefix with a mistake: repair or continue" width=45%>

- **Plant** a wrong digit at step $k$ of a correct solution, for several $k$
- At **later** positions, read the teacher's probability of the **correct** and the **planted** value: **repair** if correct wins, **continue** if planted wins
- Or let it finish greedily; compare the **final answer**
- Report $N$ (the number of prefixes tried) and the **share** of each outcome
  - add an **unplanted control**: the same solution with no planted error
- Run it with and without the extra context (hints in the teacher's prompt, RLLM-12): is the context **correct**?
- **OPD link**, if RLLM-12's premise holds: a repairing teacher shows the student what to do next. A continuing one drags it along
- *Next: checking a reward*


## Reward: why a reward must see the mistake

**Why this section?** A reward judges a whole answer. Can it see an earlier mistake?

> **A reward that cannot see a mistake cannot punish it. Whether RL fixes what SFT leaves depends on the reward's design.**

- **SFT demonstrations contain only correct prefixes**: the **exposure bias** of RLLM-7
- Take an answer with two parts: **$a$** first, then **$b$**, an independent quantity with its own true value
  - the answer reports the combined number $c=5a+b$
- Every demonstration has the **true** $a$ before $b$, so SFT never shows **what to do with $b$ after a wrong $a$**
- Does **RL** fix that? The toy on the next slide says it depends on the reward
- *Next: a toy with two rewards*


## Reward: a toy, two rewards for one two-part answer

A reward that sees only the combined $c$ meets a **wrong** $a=5$. True pair $(a,b)=(4,7)$, true $c=27$. The wrong $a$ adds $+5$.

**Joint reward**: judges only the combined $c$. **Per-value reward**: judges $a$ and $b$ separately.

| student's $b$ | $c$ | **joint** reward | **per-value** reward |
|---|---|---|---|
| 2 | 27 | **1** (accepted) | 0 (both wrong) |
| **7** (true) | 32 | 0 | **0.5** (only $b$ right) |

- **Joint**: 1 if $|c-27|\le1$ (a tolerance of 1). It accepts $b=1,2,3$
  - the $+5$ from a wrong $a$ is **cancelled** by lowering $b$, since $c=5a+b$
- **Per-value**: 0.5 for each of $a$ and $b$ exactly right. Its price: the designer must know **every true intermediate value**
- The joint reward **accepts a wrong $b$** and **rejects the true $b$**
- **This toy shows**: here, seeing the mistake depends on the **reward design**, not on RL
- *Next: two short lists that probe any reward*


## Reward: a reward probe, two short lists

Two lists for **any** reward. In the toy, the **joint** reward accepts offsetting errors; the **per-value** reward credits each value.

| List | Joint reward | Per-value reward |
|---|---|---|
| **wrong but passing** | **two errors that cancel**: the wrong $b$ above | a value right **by luck**, or credit for a **partial** match |
| **right but failing** | a correct value in a **format** the reward cannot parse (`7 units` for `7`): a reward **fault**, not gaming | the same |

- Count how many land in each list; both should be **short**
  - if either is long, the reward is not a safe stand-in
- For a **reward model**: held-out pairs with a known better answer. Where does it misjudge?
- A reward is an **assumption** the method inherits
- *Next: questions to predict*


## Think about

1. **Predict first**: a mean gap of 3 points, three seeds each, spread $s=4$. Compute the standard error of the difference and conclude.
2. A friend says "**method X beat method Y** in my run". Which **two** questions do you ask first?
3. Change the toy to true $(a,b)=(3,9)$ and a wrong $a=4$. Which $b$ values does the **joint** reward accept?
4. Choose **one** check from this session and describe a **toy** where it would show something. What would **surprise** you?


## What we established, and what it sets up

- **Established**: trust a comparison only after **checks** (noise, collapse, teacher, reward). Across Part 2, the **three axes** place every method
- **Limit of this session**:
  - checks say whether a comparison **can** be trusted, not which method is best
  - RLLM-12's **open premise** stays a **hypothesis** until a probe settles it
  - **the whole series**: placements are our reading
  - every number in Part 2 is illustrative: invented, or computed from invented inputs (RLLM-1 to RLLM-6 cite published results)
- **Completes** Part 2 and the RLLM series: RLLM-14 sets up no further session
- **You can now**:
  - place a method by the three axes (whose samples, who supplies the weights, how fine),
  - pick a first candidate by what you hold,
  - and check a comparison, a teacher and a reward before trusting them


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Henderson et al. (2018), *Deep Reinforcement Learning that Matters*](https://arxiv.org/abs/1709.06560): abstract and introduction: why a single run is not a comparison (seed and implementation variance in deep RL)
  - [Xu et al. (2024), *Is DPO Superior to PPO for LLM Alignment? A Comprehensive Study*](https://arxiv.org/abs/2404.10719): abstract and introduction: how one comparison of two methods is designed
- **Notes**
  - The second paper argues for a recommendation, whereas this series ranks nothing: read it as one team's findings.
- **Additional** (2 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


### Aside, justification: why the range check is only a first check

*For your own review; we will not work through it in session.*

1. Let each method's result over seeds have true **mean** $\mu$ and **standard
   deviation** $s$; from $n$ seeds we only have the estimates $\hat{\mu}$ and
   $\hat{s}$. *(The spread.)*
2. With $n$ seeds, the estimated **mean** of a method is uncertain by about
   $\hat{s}/\sqrt{n}$. *(Averaging shrinks the noise.)*
3. A gap between two means is **distinguishable** from noise when it is
   several multiples of $\sqrt{\hat{s}_A^2/n + \hat{s}_B^2/n}$ (for equal $\hat{s}$: $\sqrt{2\hat{s}^2/n}$). *(The noise on a
   difference of two independent means.)*
4. "The ranges do not overlap" is a **cruder** screen for small $n$. *(It ignores the averaging; it is not a test. More seeds widen the observed range, so it gets harder to meet.)*
5. With $n$ very small, **both** are weak: say the number of seeds.

**Result, in words.** The range check is a **first filter**, not a test.
